In [1]:
import s3fs
fs = s3fs.S3FileSystem(anon=True)

# Browse the structure
print(fs.ls("nasa-surya-bench/"))

['nasa-surya-bench/2010', 'nasa-surya-bench/2011', 'nasa-surya-bench/2012', 'nasa-surya-bench/2013', 'nasa-surya-bench/2014', 'nasa-surya-bench/2015', 'nasa-surya-bench/2016', 'nasa-surya-bench/2017', 'nasa-surya-bench/2018', 'nasa-surya-bench/2019', 'nasa-surya-bench/2020', 'nasa-surya-bench/2021', 'nasa-surya-bench/2022', 'nasa-surya-bench/2023', 'nasa-surya-bench/2024', 'nasa-surya-bench/index.html']


In [2]:
from datetime import datetime
import pandas as pd
import numpy as np

df = pd.read_csv("SURYA_final.csv")
sep_dates = df['window_begin'].tolist()
print(len(sep_dates))
s3_paths = []

for date_str in sep_dates:
    dt = pd.to_datetime(date_str)
    prefix = f"nasa-surya-bench/{dt.year}/{dt.month:02d}/"
    try:
        files = fs.ls(prefix)
        matches = [f for f in files if f"/{dt.strftime('%Y%m%d')}" in f]
        s3_paths.append(f"s3://{matches[0]}" if matches else np.nan)
    except Exception:
        s3_paths.append(np.nan)

df['path'] = s3_paths
print(df[['window_begin', 'path','OSEP_label','Flare_log_Strength_max']])

25


   window_begin                                            path  OSEP_label  \
0     3/20/2015  s3://nasa-surya-bench/2015/03/20150320_0000.nc           0   
1      2/9/2012  s3://nasa-surya-bench/2012/02/20120209_0000.nc           0   
2     3/18/2023  s3://nasa-surya-bench/2023/03/20230318_0000.nc           0   
3    10/26/2021  s3://nasa-surya-bench/2021/10/20211026_0000.nc           0   
4     6/21/2015  s3://nasa-surya-bench/2015/06/20150621_0000.nc           1   
5     12/5/2020  s3://nasa-surya-bench/2020/12/20201205_0000.nc           0   
6     1/12/2017  s3://nasa-surya-bench/2017/01/20170112_0000.nc           0   
7     9/10/2017  s3://nasa-surya-bench/2017/09/20170910_0000.nc           1   
8     6/21/2018  s3://nasa-surya-bench/2018/06/20180621_0000.nc           0   
9      6/7/2011  s3://nasa-surya-bench/2011/06/20110607_0000.nc           1   
10   12/12/2022  s3://nasa-surya-bench/2022/12/20221212_0000.nc           0   
11     4/2/2022  s3://nasa-surya-bench/2022/04/20220

In [3]:
df = df.rename(columns={"OSEP_label": "SEP"})

df['window_begin'] = pd.to_datetime(df['window_begin']).dt.strftime('%Y-%m-%d %H:%M:%S')

df = df.rename(columns={"Flare_log_Strength_max": "flare_strength"})

df = df.rename(columns={"window_begin": "timestep"})

df['present']=1

df.loc[df['path'].isna(), 'present'] = 0

In [5]:
df.to_csv("SURYA_data_v2.csv",index=False)

In [3]:
import pandas as pd

df=pd.read_csv("SURYA_data_v2.csv")

df = df.drop(columns=['Unnamed: 0'])



In [4]:


df.to_csv("SURYA_data_v2.csv",index=False)